In [27]:
import pandas as pd
df = pd.read_csv("star_classification.csv")
df.head()

,obj_ID,alpha,delta,u,g,r,i,z,run_ID,rerun_ID,cam_col,field_ID,spec_obj_ID,class,redshift,plate,MJD,fiber_ID
0,1.237661e+18,135.689107,32.494632,23.87882,22.27530,20.39501,19.16573,18.79371,3606,301,2,79,6.543777e+18,GALAXY,0.634794,5812,56354,171
1,1.237665e+18,144.826101,31.274185,24.77759,22.83188,22.58444,21.16812,21.61427,4518,301,5,119,1.176014e+19,GALAXY,0.779136,10445,58158,427
2,1.237661e+18,142.188790,35.582444,25.26307,22.66389,20.60976,19.34857,18.94827,3606,301,2,120,5.152200e+18,GALAXY,0.644195,4576,55592,299
3,1.237663e+18,338.741038,-0.402828,22.13682,23.77656,21.61162,20.50454,19.25010,4192,301,3,214,1.030107e+19,GALAXY,0.932346,9149,58039,775
4,1.237680e+18,345.282593,21.183866,19.43718,17.58028,16.49747,15.97711,15.54461,8102,301,3,137,6.891865e+18,GALAXY,0.116123,6121,56187,842


In [28]:
print("Shape:", df.shape)
print()
print(df["class"].value_counts())
print()
df.info()

Shape: (100000, 18)

class
GALAXY    59445
STAR      21594
QSO       18961
Name: count, dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 18 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   obj_ID       100000 non-null  float64
 1   alpha        100000 non-null  float64
 2   delta        100000 non-null  float64
 3   u            100000 non-null  float64
 4   g            100000 non-null  float64
 5   r            100000 non-null  float64
 6   i            100000 non-null  float64
 7   z            100000 non-null  float64
 8   run_ID       100000 non-null  int64  
 9   rerun_ID     100000 non-null  int64  
 10  cam_col      100000 non-null  int64  
 11  field_ID     100000 non-null  int64  
 12  spec_obj_ID  100000 non-null  float64
 13  class        100000 non-null  object 
 14  redshift     100000 non-null  float64
 15  plate        100000 non-null  int64  
 16  MJD      

In [29]:

features = ["u", "g", "r", "i", "z"]
X = df[features]
y = df["class"]
print("Features (X):", X.shape)
print("Target (y):", y.shape)
X.head()

Features (X): (100000, 5)
Target (y): (100000,)


,u,g,r,i,z
0,23.87882,22.27530,20.39501,19.16573,18.79371
1,24.77759,22.83188,22.58444,21.16812,21.61427
2,25.26307,22.66389,20.60976,19.34857,18.94827
3,22.13682,23.77656,21.61162,20.50454,19.25010
4,19.43718,17.58028,16.49747,15.97711,15.54461


In [30]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (80000, 5)
Test set: (20000, 5)


In [31]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Training complete!")

Training complete!


In [32]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 87.38 %


In [33]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred, labels=["GALAXY", "STAR", "QSO"])
cm_df = pd.DataFrame(cm,
    index=["true GALAXY", "true STAR", "true QSO"],
    columns=["pred GALAXY", "pred STAR", "pred QSO"])
print(cm_df)

              precision    recall  f1-score   support

      GALAXY       0.91      0.94      0.93     11889
         QSO       0.81      0.80      0.80      3792
        STAR       0.82      0.75      0.78      4319

    accuracy                           0.87     20000
   macro avg       0.85      0.83      0.84     20000
weighted avg       0.87      0.87      0.87     20000

             pred GALAXY  pred STAR  pred QSO
true GALAXY        11218        369       302
true STAR            669       3237       413
true QSO             440        331      3021


In [34]:
import pandas as pd
importances = pd.Series(model.feature_importances_, index=features)
importances = importances.sort_values(ascending=False)
print(importances)

z    0.256590
g    0.204076
u    0.203546
i    0.169897
r    0.165891
dtype: float64


In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


features_with_z = ["u", "g", "r", "i", "z", "redshift"]
X2 = df[features_with_z]
y2 = df["class"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2)

model2 = RandomForestClassifier(n_estimators=100, random_state=42)
model2.fit(X2_train, y2_train)

acc2 = accuracy_score(y2_test, model2.predict(X2_test))
print("Accuracy WITH redshift:", round(acc2 * 100, 2), "%")
print("Accuracy WITHOUT redshift (from before): ~87%")

Accuracy WITH redshift: 97.94 %
Accuracy WITHOUT redshift (from before): ~87%


In [36]:
import numpy as np
from sklearn.metrics import accuracy_score

print("Noise level (std)  ->  Accuracy")
for noise in [0.0, 0.1, 0.25, 0.5, 1.0]:

    X_test_noisy = X_test + np.random.normal(0, noise, X_test.shape)
    acc = accuracy_score(y_test, model.predict(X_test_noisy))
    print(f"      {noise:<12} ->  {round(acc*100, 2)}%")

Noise level (std)  ->  Accuracy
      0.0          ->  87.38%
      0.1          ->  83.65%
      0.25         ->  72.91%
      0.5          ->  57.11%
      1.0          ->  42.43%


In [37]:
import joblib
joblib.dump(model, "stellar_model.joblib")
print("Model saved!")

Model saved!


In [38]:
from google.colab import files
files.download("stellar_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>